In [1]:
import os
import re
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import r2_score
from ipywidgets import Dropdown, HBox, VBox, Button, Output, Text, widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")

try:
    from xbbg import blp
    HAVE_BBG = True
except Exception:
    HAVE_BBG = False

In [2]:
DATA_DIR   = r"J:\All Data\53B. Bonds Basis Cont. Mat"
BASIS_FILE = os.path.join(DATA_DIR, "Basis df5a.parquet")

# Optional extras
CDX_MEMBERS_FILE     = "CDX Members.xlsx"         # only used for the 'Family' column of the summary table
LOAD_BBG_MARKET_DATA = True                       # iBoxx HY/IG, UST 5/7/10Y, VIX, COA overlays
BBG_CACHE_FILE       = "BBG Market Data.parquet"  # used when Bloomberg is not reachable

# Fall back to a local copy if the J: drive is not mapped
if not os.path.exists(BASIS_FILE) and os.path.exists("Basis df5a.parquet"):
    BASIS_FILE = "Basis df5a.parquet"
print("Reading:", os.path.abspath(BASIS_FILE))

Reading: J:\All Data\53B. Bonds Basis Cont. Mat\Basis df5a.parquet


In [3]:
df5a = pd.read_parquet(BASIS_FILE)
ID_COLS = ["ISIN", "CUSIP", "Name", "Maturity", "Issuer Equity_red_tier_ccy_docclause"]
df5a["Maturity"] = pd.to_datetime(df5a["Maturity"])

# Parquet column prefix -> label used in the plot menus (unknown prefixes are shown with '_' -> ' ')
METRIC_LABELS = {
    "Price":             "Price",
    "Sprd":              "G-Sprd",
    "Mat_Matched_CDS":   "Mat. Matched CDS",
    "Basis":             "Basis",
    "Mat_Matched_UF100": "UF100 Mat. Matched CDS",
    "UF100_Basis":       "UF100 Basis",
    "Mat_Matched_UF500": "UF500 Mat. Matched CDS",
    "UF500_Basis":       "UF500 Basis",
    "Dur":               "Duration",
}
METRIC_ORDER = list(METRIC_LABELS.values())

# ---------- split '<prefix>_<YYYY-MM-DD>' columns by prefix
date_re = re.compile(r"^(.+)_(\d{4}-\d{2}-\d{2})$")
col_info = {}
for col in df5a.columns:
    m = date_re.match(col)
    if m:
        col_info.setdefault(m.group(1), []).append((col, m.group(2)))
for prefix in col_info:
    col_info[prefix].sort(key=lambda t: t[1])

last_update = col_info["Basis"][-1][1]

# ---------- bond labels, same style as v13: 'TOL 3.8 11/01/29' -> 'TOL 3.8% due 29'
# If two bonds of one issuer would get the same label, the full date is used instead: 'AAPL 4% due 05/10/28'
name_re = re.compile(r"^(?P<head>.+) (?P<mm>\d{2})/(?P<dd>\d{2})/(?P<yy>\d{2,4})$")

def make_labels(frame):
    short, full = {}, {}
    for isin, name in zip(frame["ISIN"], frame["Name"]):
        m = name_re.match(name)
        short[isin] = f"{m['head']}% due {m['yy']}" if m else name
        full[isin]  = f"{m['head']}% due {m['mm']}/{m['dd']}/{m['yy']}" if m else name
    counts = pd.Series(short).value_counts()
    labels = {i: (full[i] if counts[s] > 1 else s) for i, s in short.items()}
    counts = pd.Series(labels).value_counts()
    return {i: (f"{l} [{i}]" if counts[l] > 1 else l) for i, l in labels.items()}

bonds = df5a[ID_COLS].copy()
bonds["Label"]  = bonds["ISIN"].map(make_labels(df5a))
bonds["Issuer"] = bonds["Label"].str.split(" ", n=1).str[0]
isin_to_label   = dict(zip(bonds["ISIN"], bonds["Label"]))

# ---------- wide time series: one row per date, one column per '<bond label> <metric>'
by_isin = df5a.set_index("ISIN")
frames, col_meta = [], {}
for prefix, items in col_info.items():
    metric = METRIC_LABELS.get(prefix, prefix.replace("_", " "))
    block = by_isin[[c for c, _ in items]].T
    block.index = pd.to_datetime([d for _, d in items])
    block.columns = [f"{isin_to_label[i]} {metric}" for i in block.columns]
    frames.append(block)

ts = pd.concat(frames, axis=1).sort_index().astype(float)
ts = ts.dropna(how="all").dropna(axis=1, how="all")
ts.index.name = "Date"

mat_by_label = dict(zip(bonds["Label"], bonds["Maturity"]))
for prefix in col_info:
    metric = METRIC_LABELS.get(prefix, prefix.replace("_", " "))
    rank = METRIC_ORDER.index(metric) if metric in METRIC_ORDER else len(METRIC_ORDER)
    for label in bonds["Label"]:
        col = f"{label} {metric}"
        if col in ts.columns:
            col_meta[col] = (label.split(" ", 1)[0], mat_by_label[label], rank, col)

print(f"{len(bonds):,} bonds · {bonds['Issuer'].nunique()} issuers · "
      f"{ts.index[0].date()} to {ts.index[-1].date()} · last basis date {last_update}")
print("Metrics:", ", ".join(METRIC_LABELS.get(p, p) for p in col_info))

3,814 bonds · 343 issuers · 2021-10-28 to 2026-10-27 · last basis date 2026-10-26
Metrics: Mat. Matched CDS, G-Sprd, Basis, Price, UF100 Mat. Matched CDS, UF500 Mat. Matched CDS, UF100 Basis, UF500 Basis


In [4]:
# ---------- Family map from CDX Members (optional)
def load_family_map(paths):
    res_codes = {"Full Restructuring": "CR14", "Modified Restructuring": "MR14",
                 "Modified-Modified Restructurin": "MM14", "No Restructuring": "XR14"}
    path = next((p for p in paths if os.path.exists(p)), None)
    if path is None:
        return {}
    sheets = pd.read_excel(path, sheet_name=None)
    fam = {}
    for sheet in ["IG", "HY", "EUR_IG", "EUR_HY", "SNRFIN", "SUBFIN", "Extras"]:
        if sheet not in sheets:
            continue
        dq = sheets[sheet].copy()
        if "Family" not in dq.columns:
            dq["Family"] = sheet
        if "tier" not in dq.columns:
            dq["tier"] = "SNRFOR" if sheet != "SUBFIN" else "SUBLT2"
        dq["ccy"] = dq["Family"].apply(lambda x: "USD" if x in ["IG", "HY"] else "EUR")
        keys = (dq["Issuer Equity"].astype(str) + "_" + dq["Actual RED Code"].astype(str) + "_" +
                dq["tier"].astype(str) + "_" + dq["ccy"].astype(str) + "_" +
                dq["Restructuring"].map(res_codes).astype(str))
        fam.update(dict(zip(keys, dq["Family"])))
    return fam

f_dict = load_family_map([CDX_MEMBERS_FILE, os.path.join(DATA_DIR, CDX_MEMBERS_FILE)])

# ---------- Z-scores and 1Y high/low on Basis (as in v13)
basis_hist = by_isin[[c for c, _ in col_info["Basis"]]].T
basis_hist.index = pd.to_datetime([d for _, d in col_info["Basis"]])

stats = {}
for months in (3, 6, 12):
    p = months * 22
    roll = basis_hist.rolling(p, min_periods=int(0.6 * p))
    stats[f"ZScore_{months}M"] = ((basis_hist - roll.mean()) / roll.std()).iloc[-1]
stats["Basis 1Y High"] = basis_hist.iloc[-12 * 22:].max()
stats["Basis 1Y Low"]  = basis_hist.iloc[-12 * 22:].min()
stats = pd.DataFrame(stats)

def latest(prefix):
    col = f"{prefix}_{last_update}"
    return by_isin[col] if col in by_isin.columns else pd.Series(np.nan, index=by_isin.index)

summary = pd.DataFrame({
    "Name":             by_isin["Name"],
    "Family":           by_isin["Issuer Equity_red_tier_ccy_docclause"].map(f_dict).fillna("n/a"),
    "CDS RED Code":     by_isin["Issuer Equity_red_tier_ccy_docclause"].str.split("_").str[1],
    "Maturity":         by_isin["Maturity"].dt.date,
    "Price":            latest("Price"),
    "Sprd":             latest("Sprd"),
    "Mat. Matched CDS": latest("Mat_Matched_CDS"),
    "Basis":            latest("Basis"),
    "UF100 Basis":      latest("UF100_Basis"),
    "UF500 Basis":      latest("UF500_Basis"),
}).join(stats)

summary["_issuer"] = summary["Name"].str.split(" ").str[0]
summary = (summary.sort_values(["_issuer", "Maturity"]).drop(columns="_issuer")
                  .set_index("Name").round(2))
df7 = summary   # same name as in v13
print(f"Summary as of {last_update}" + ("" if f_dict else "  (CDX Members file not found, Family = n/a)"))
summary

Summary as of 2026-10-26


,Family,CDS RED Code,Maturity,Price,Sprd,Mat. Matched CDS,Basis,UF100 Basis,UF500 Basis,ZScore_3M,ZScore_6M,ZScore_12M,Basis 1Y High,Basis 1Y Low
Name,,,,,,,,,,,,,,
AALLN 4 ½ 09/15/28,EUR_IG,037CDU,2028-09-15,101.45,25.21,24.79,-0.42,0.09,-7.31,0.96,0.72,0.16,7.81,-168.78
AALLN 3 ¾ 06/15/29,EUR_IG,037CDU,2029-06-15,99.60,47.41,38.82,-8.59,-1.90,-12.08,-1.80,-2.54,-1.05,3.07,-14.47
AALLN 5 03/15/31,EUR_IG,037CDU,2031-03-15,102.95,77.08,75.53,-1.55,1.99,-14.31,0.01,-0.53,-0.70,18.66,-7.72
AALLN 4 ⅛ 03/15/32,EUR_IG,037CDU,2032-03-15,98.90,92.68,96.02,3.34,-1.24,-20.74,0.74,0.10,-0.50,23.23,-6.62
AALLN 4 ¾ 09/21/32,EUR_IG,037CDU,2032-09-21,101.82,95.81,105.01,9.21,2.20,-18.83,0.53,0.00,-0.56,22.94,-0.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZFFNGR 7 06/12/30,EUR_HY,NWGDFJ,2030-06-12,103.40,241.81,236.46,-5.35,8.02,-5.16,-1.56,-1.93,-0.61,39.57,-33.89
ZIGGO 3 ⅜ 02/28/30,EUR_HY,NWHB95,2030-02-28,65.33,1413.23,1441.21,27.98,-3.16,-12.55,-0.17,0.25,0.68,178.05,-151.04
ZIGGO 6 ⅛ 11/15/32,EUR_HY,NWHB95,2032-11-15,58.70,1404.73,1474.00,69.27,0.01,-12.01,0.45,0.48,0.16,126.27,-83.61


In [5]:
df_bbg = pd.DataFrame()

if LOAD_BBG_MARKET_DATA:
    try:
        if not HAVE_BBG:
            raise ImportError("xbbg not installed")
        tickers_bbg = ['IBOXHY INDEX', 'IBOXIG INDEX']
        fields_bbg  = ['PX_LAST', 'CONTRBTD_ZSPREAD', 'OAS_SOVEREIGN_CURVE', 'OPTION_ADJ_DURATION_SOV_CRV', 'YIELD_TO_MATURITY']
        df_bbg = blp.bdh(tickers=tickers_bbg, flds=fields_bbg, start_date="2000-01-01",
                         end_date=datetime.now().strftime('%Y-%m-%d'))
        list1_bbg = ['TR', 'Z-Sprd', 'OAS', 'Dur', 'Yield']
        df_bbg.columns = ['BBG iBoxx HY ' + i for i in list1_bbg] + ['BBG iBoxx IG ' + i for i in list1_bbg]
        df_bbg = df_bbg.drop(['BBG iBoxx HY Dur', 'BBG iBoxx IG Yield'], axis=1).astype(float)  # first is wrong, second incomplete

        x = blp.bdh(tickers=['USGG5YR INDEX', 'USGG7YR INDEX', 'USGG10YR INDEX'], flds='PX_LAST', start_date=df_bbg.index[0])
        x.columns = ['BBG US 5Y', 'BBG US 7Y', 'BBG US 10Y']
        df_bbg = pd.concat([df_bbg, x], axis=1)

        x = blp.bdh(tickers=['COA Comdty', 'VIX Index'], flds='PX_LAST', start_date=df_bbg.index[0])
        x.columns = ['BBG COA Comdty', 'BBG VIX Index']
        df_bbg = pd.concat([df_bbg, x.sort_index()], axis=1)

        df_bbg.index = pd.to_datetime(df_bbg.index)
        df_bbg.to_parquet(BBG_CACHE_FILE)
    except Exception as err:
        if os.path.exists(BBG_CACHE_FILE):
            df_bbg = pd.read_parquet(BBG_CACHE_FILE)
            print(f"Bloomberg unavailable ({err}); using cache {BBG_CACHE_FILE} to {df_bbg.index[-1].date()}")
        else:
            print(f"Bloomberg unavailable ({err}); plotting bond data only")

df = pd.concat([ts, df_bbg], axis=1).sort_index() if len(df_bbg) else ts.copy()
df.index = pd.to_datetime(df.index)

In [6]:
# Preferred default bonds per issuer (carried over from v13). Entries that no longer exist are
# ignored, and issuers not listed fall back to the bond maturing closest to 5Y with a live Basis.
default_values_dict = \
{'TOL': ['3.8% due 29 Mat. Matched CDS', '3.8% due 29 Basis'],
 'PUBFP': ['1 ¾% due 31 Mat. Matched CDS', '1 ¾% due 31 Basis'],
 'ZIGGO': ['3 ⅜% due 30 Mat. Matched CDS', '3 ⅜% due 30 Basis'],
 'CAR': ['8% due 31 Mat. Matched CDS', '8% due 31 Basis'],
 'WFRD': ['8 ⅝% due 30 Mat. Matched CDS', '8 ⅝% due 30 Basis'],
 'MPW': ['3 ½% due 31 Mat. Matched CDS', '3 ½% due 31 Basis'],
 'CHTR': ['4 ¾% due 30 Mat. Matched CDS', '4 ¾% due 30 Basis'],
 'TSCOLN': ['4 ¼% due 31 Mat. Matched CDS', '4 ¼% due 31 Basis'],
 'HOUS': ['5 ¼% due 30 Mat. Matched CDS', '5 ¼% due 30 Basis'],
 'SIRI': ['4 ⅛% due 30 Mat. Matched CDS', '4 ⅛% due 30 Basis'],
 'TELEFO': ['5 ⅞% due 33 Mat. Matched CDS', '5 ⅞% due 33 Basis'],
 'BMW': ['0 ⅞% due 32 Mat. Matched CDS', '0 ⅞% due 32 Basis'],
 'SIEGR': ['1 ⅜% due 30 Mat. Matched CDS', '1 ⅜% due 30 Basis'],
 'CSCHLD': ['5 ¾% due 30 Mat. Matched CDS', '5 ¾% due 30 Basis'],
 'SPG': ['2.45% due 29 Mat. Matched CDS', '2.45% due 29 Basis'],
 'ALVGR': ['1 ⅜% due 31 Mat. Matched CDS', '1 ⅜% due 31 Basis'],
 'RGCARE': ['10% due 32 Mat. Matched CDS', '10% due 32 Basis'],
 'SZUGR': ['4 ⅛% due 32 Mat. Matched CDS', '4 ⅛% due 32 Basis'],
 'VW': ['3.3% due 33 Mat. Matched CDS', '3.3% due 33 Basis'],
 'SPMIM': ['4 ⅞% due 30 Mat. Matched CDS', '4 ⅞% due 30 Basis'],
 'GSK': ['1 ⅜% due 29 Mat. Matched CDS', '1 ⅜% due 29 Basis'],
 'TEVA': ['8 ⅛% due 31 Mat. Matched CDS', '8 ⅛% due 31 Basis'],
 'HLT': ['4 ⅞% due 30 Mat. Matched CDS', '4 ⅞% due 30 Basis'],
 'VST': ['7 ¾% due 31 Mat. Matched CDS', '7 ¾% due 31 Basis'],
 'CMACG': ['5% due 31 Mat. Matched CDS', '5% due 31 Basis'],
 'ALIANT': ['5 ⅞% due 29 Mat. Matched CDS', '5 ⅞% due 29 Basis'],
 'DUFNSW': ['4 ¾% due 31 Mat. Matched CDS', '4 ¾% due 31 Basis'],
 'AXL': ['5% due 29 Mat. Matched CDS', '5% due 29 Basis'],
 'IMBLN': ['3 ⅞% due 34 Mat. Matched CDS', '3 ⅞% due 34 Basis'],
 'ZFFNGR': ['3% due 29 Mat. Matched CDS', '3% due 29 Basis'],
 'TMUS': ['2 ⅞% due 31 Mat. Matched CDS', '2 ⅞% due 31 Basis'],
 'URI': ['5 ¼% due 30 Mat. Matched CDS', '5 ¼% due 30 Basis'],
 'DOW': ['7 ⅜% due 29 Mat. Matched CDS', '7 ⅜% due 29 Basis'],
 'NCLH': ['6 ¼% due 30 Mat. Matched CDS', '6 ¼% due 30 Basis'],
 'M': ['4 ½% due 34 Mat. Matched CDS', '4 ½% due 34 Basis'],
 'BPLN': ['1.231% due 31 Mat. Matched CDS', '1.231% due 31 Basis'],
 'JCI': ['1 ¾% due 30 Mat. Matched CDS', '1 ¾% due 30 Basis'],
 'AALLN': ['4 ¾% due 32 Mat. Matched CDS', '4 ¾% due 32 Basis'],
 'OGN': ['5 ⅛% due 31 Mat. Matched CDS', '5 ⅛% due 31 Basis'],
 'LOUDRE': ['3 ½% due 31 Mat. Matched CDS', '3 ½% due 31 Basis'],
 'MEDIND': ['5 ¼% due 29 Mat. Matched CDS', '5 ¼% due 29 Basis'],
 'HOLNSW': ['1 ¾% due 29 Mat. Matched CDS', '1 ¾% due 29 Basis'],
 'GLENLN': ['1 ¼% due 33 Mat. Matched CDS', '1 ¼% due 33 Basis'],
 'NRUC': ['8% due 32 Mat. Matched CDS', '8% due 32 Basis'],
 'LINTA': ['4% due 29 Mat. Matched CDS', '4% due 29 Basis'],
 'CARLB': ['0 ⅝% due 30 Mat. Matched CDS', '0 ⅝% due 30 Basis'],
 'RIG': ['7 ½% due 31 Mat. Matched CDS', '7 ½% due 31 Basis'],
 'REPSM': ['2 ⅝% due 30 Mat. Matched CDS', '2 ⅝% due 30 Basis'],
 'NBR': ['9 ⅛% due 30 Mat. Matched CDS', '9 ⅛% due 30 Basis'],
 'HILCRP': ['6% due 31 Mat. Matched CDS', '6% due 31 Basis'],
 'CE': ['6.629% due 32 Mat. Matched CDS', '6.629% due 32 Basis'],
 'HST': ['3 ⅜% due 29 Mat. Matched CDS', '3 ⅜% due 29 Basis'],
 'HRB': ['3 ⅞% due 30 Mat. Matched CDS', '3 ⅞% due 30 Basis'],
 'BATSLN': ['2 ¼% due 30 Mat. Matched CDS', '2 ¼% due 30 Basis'],
 'TRPCN': ['5.6% due 34 Mat. Matched CDS', '5.6% due 34 Basis'],
 'BRITEL': ['1 ⅛% due 29 Mat. Matched CDS', '1 ⅛% due 29 Basis'],
 'RDSALN': ['0 ½% due 31 Mat. Matched CDS', '0 ½% due 31 Basis'],
 'BMCAUS': ['4 ⅜% due 30 Mat. Matched CDS', '4 ⅜% due 30 Basis'],
 'NEE': ['2 ¾% due 29 Mat. Matched CDS', '2 ¾% due 29 Basis'],
 'HTZ': ['5% due 29 Mat. Matched CDS', '5% due 29 Basis'],
 'VMED': ['3 ¾% due 30 Mat. Matched CDS', '3 ¾% due 30 Basis'],
 'VLVY': ['3% due 30 Mat. Matched CDS', '3% due 30 Basis'],
 'HCA': ['3 ½% due 30 Mat. Matched CDS', '3 ½% due 30 Basis'],
 'AIFP': ['0 ⅝% due 30 Mat. Matched CDS', '0 ⅝% due 30 Basis'],
 'NTGYSM': ['1 ⅞% due 29 Mat. Matched CDS', '1 ⅞% due 29 Basis'],
 'ABIBB': ['3 ¼% due 33 Mat. Matched CDS', '3 ¼% due 33 Basis'],
 'ACFP': ['3 ⅞% due 31 Mat. Matched CDS', '3 ⅞% due 31 Basis'],
 'ACAFP': ['2% due 30 Mat. Matched CDS', '2% due 30 Basis'],
 'AEP': ['2.3% due 30 Mat. Matched CDS', '2.3% due 30 Basis'],
 'AES': ['3.95% due 30 Mat. Matched CDS', '3.95% due 30 Basis'],
 'AFFP': ['3 ¾% due 30 Mat. Matched CDS', '3 ¾% due 30 Basis'],
 'OMF': ['5 ⅜% due 29 Mat. Matched CDS', '5 ⅜% due 29 Basis'],
 'AIG': ['3 ⅞% due 35 Mat. Matched CDS', '3 ⅞% due 35 Basis'],
 'AIRFP': ['2 ⅛% due 29 Mat. Matched CDS', '2 ⅛% due 29 Basis'],
 'AKZANA': ['1 ⅝% due 30 Mat. Matched CDS', '1 ⅝% due 30 Basis'],
 'ALL': ['5.35% due 33 Mat. Matched CDS', '5.35% due 33 Basis'],
 'ALLY': ['8% due 31 Mat. Matched CDS', '8% due 31 Basis'],
 'ALOFP': ['0 ½% due 30 Mat. Matched CDS', '0 ½% due 30 Basis'],
 'AMD': ['2 ⅜% due 30 Mat. Matched CDS', '2 ⅜% due 30 Basis'],
 'AMGN': ['2.45% due 30 Mat. Matched CDS', '2.45% due 30 Basis'],
 'AMZN': ['4.8% due 34 Mat. Matched CDS', '4.8% due 34 Basis'],
 'APA': ['4 ¼% due 30 Mat. Matched CDS', '4 ¼% due 30 Basis'],
 'ARW': ['2.95% due 32 Mat. Matched CDS', '2.95% due 32 Basis'],
 'ASH': ['3 ⅜% due 31 Mat. Matched CDS', '3 ⅜% due 31 Basis'],
 'ATLIM': ['4 ½% due 30 Mat. Matched CDS', '4 ½% due 30 Basis'],
 'AVGO': ['4.15% due 30 Mat. Matched CDS', '4.15% due 30 Basis'],
 'AVNT': ['7 ⅛% due 30 Mat. Matched CDS', '7 ⅛% due 30 Basis'],
 'AVT': ['3% due 31 Mat. Matched CDS', '3% due 31 Basis'],
 'AXP': ['4.42% due 33 Mat. Matched CDS', '4.42% due 33 Basis'],
 'AZN': ['3 ¾% due 32 Mat. Matched CDS', '3 ¾% due 32 Basis'],
 'AZO': ['4% due 30 Mat. Matched CDS', '4% due 30 Basis'],
 'BA': ['6 ⅛% due 33 Mat. Matched CDS', '6 ⅛% due 33 Basis'],
 'BALL': ['2 ⅞% due 30 Mat. Matched CDS', '2 ⅞% due 30 Basis'],
 'BACR': ['1 ⅛% due 31 Mat. Matched CDS', '1 ⅛% due 31 Basis'],
 'BASGR': ['3% due 33 Mat. Matched CDS', '3% due 33 Basis'],
 'BAX': ['3.95% due 30 Mat. Matched CDS', '3.95% due 30 Basis'],
 'BAYNGR': ['2 ⅛% due 29 Mat. Matched CDS', '2 ⅛% due 29 Basis'],
 'BBDBCN': ['7.45% due 34 Mat. Matched CDS', '7.45% due 34 Basis'],
 'BBVASM': ['5 ¾% due 33 Mat. Matched CDS', '5 ¾% due 33 Basis'],
 'BBWI': ['6 ⅝% due 30 Mat. Matched CDS', '6 ⅝% due 30 Basis'],
 'BBY': ['1.95% due 30 Mat. Matched CDS', '1.95% due 30 Basis'],
 'BHCCN': ['5 ¼% due 30 Mat. Matched CDS', '5 ¼% due 30 Basis'],
 'BMY': ['1.45% due 30 Mat. Matched CDS', '1.45% due 30 Basis'],
 'BNFP': ['0.52% due 30 Mat. Matched CDS', '0.52% due 30 Basis'],
 'BNP': ['2 ⅜% due 30 Mat. Matched CDS', '2 ⅜% due 30 Basis'],
 'BRK': ['6 ½% due 34 Mat. Matched CDS', '6 ½% due 34 Basis'],
 'BSX': ['2.65% due 30 Mat. Matched CDS', '2.65% due 30 Basis'],
 'BERTEL': ['1 ½% due 30 Mat. Matched CDS', '1 ½% due 30 Basis'],
 'BWA': ['5.4% due 34 Mat. Matched CDS', '5.4% due 34 Basis'],
 'BYD': ['4 ¾% due 31 Mat. Matched CDS', '4 ¾% due 31 Basis'],
 'BZH': ['7 ¼% due 29 Mat. Matched CDS', '7 ¼% due 29 Basis'],
 'CAFP': ['2 ⅜% due 29 Mat. Matched CDS', '2 ⅜% due 29 Basis'],
 'CAG': ['8 ¼% due 30 Mat. Matched CDS', '8 ¼% due 30 Basis'],
 'CAH': ['5.45% due 34 Mat. Matched CDS', '5.45% due 34 Basis'],
 'CMZB': ['4% due 30 Mat. Matched CDS', '4% due 30 Basis'],
 'CCL': ['6 ⅛% due 33 Mat. Matched CDS', '6 ⅛% due 33 Basis'],
 'CIVI': ['8 ¾% due 31 Mat. Matched CDS', '8 ¾% due 31 Basis'],
 'CLF': ['4 ⅞% due 31 Mat. Matched CDS', '4 ⅞% due 31 Basis'],
 'CLNXSM': ['1 ¾% due 30 Mat. Matched CDS', '1 ¾% due 30 Basis'],
 'CMCSA': ['7.05% due 33 Mat. Matched CDS', '7.05% due 33 Basis'],
 'CNQCN': ['7.2% due 32 Mat. Matched CDS', '7.2% due 32 Basis'],
 'COF': ['2.7% due 30 Mat. Matched CDS', '2.7% due 30 Basis'],
 'CONGR': ['3 ½% due 29 Mat. Matched CDS', '3 ½% due 29 Basis'],
 'COXENT': ['4.8% due 35 Mat. Matched CDS', '4.8% due 35 Basis'],
 'CPB': ['2 ⅜% due 30 Mat. Matched CDS', '2 ⅜% due 30 Basis'],
 'CPGLN': ['3 ¼% due 31 Mat. Matched CDS', '3 ¼% due 31 Basis'],
 'CPN': ['5% due 31 Mat. Matched CDS', '5% due 31 Basis'],
 'AXASA': ['3 ¾% due 30 Mat. Matched CDS', '3 ¾% due 30 Basis'],
 'CSCO': ['4.95% due 31 Mat. Matched CDS', '4.95% due 31 Basis'],
 'CSTM': ['5 ⅜% due 32 Mat. Matched CDS', '5 ⅜% due 32 Basis'],
 'CSX': ['2.4% due 30 Mat. Matched CDS', '2.4% due 30 Basis'],
 'CVS': ['4 ⅞% due 35 Mat. Matched CDS', '4 ⅞% due 35 Basis'],
 'CZR': ['4 ⅝% due 29 Mat. Matched CDS', '4 ⅝% due 29 Basis'],
 'D': ['6.3% due 33 Mat. Matched CDS', '6.3% due 33 Basis'],
 'DAL': ['3 ¾% due 29 Mat. Matched CDS', '3 ¾% due 29 Basis'],
 'DANBNK': ['1 ½% due 30 Mat. Matched CDS', '1 ½% due 30 Basis'],
 'DB': ['5 ⅝% due 31 Mat. Matched CDS', '5 ⅝% due 31 Basis'],
 'DE': ['7 ⅛% due 31 Mat. Matched CDS', '7 ⅛% due 31 Basis'],
 'DGFP': ['1 ¾% due 30 Mat. Matched CDS', '1 ¾% due 30 Basis'],
 'DGX': ['2.95% due 30 Mat. Matched CDS', '2.95% due 30 Basis'],
 'DHI': ['5% due 34 Mat. Matched CDS', '5% due 34 Basis'],
 'DHR': ['2.6% due 29 Mat. Matched CDS', '2.6% due 29 Basis'],
 'DIS': ['7% due 32 Mat. Matched CDS', '7% due 32 Basis'],
 'DRI': ['6.3% due 33 Mat. Matched CDS', '6.3% due 33 Basis'],
 'DT': ['7 ½% due 33 Mat. Matched CDS', '7 ½% due 33 Basis'],
 'DVA': ['4 ⅝% due 30 Mat. Matched CDS', '4 ⅝% due 30 Basis'],
 'DVN': ['7 ⅞% due 31 Mat. Matched CDS', '7 ⅞% due 31 Basis'],
 'EDF': ['5 ⅝% due 33 Mat. Matched CDS', '5 ⅝% due 33 Basis'],
 'EDPPL': ['1 ⅞% due 29 Mat. Matched CDS', '1 ⅞% due 29 Basis'],
 'ELTLX': ['2 ½% due 30 Mat. Matched CDS', '2 ½% due 30 Basis'],
 'EMN': ['5 ¾% due 33 Mat. Matched CDS', '5 ¾% due 33 Basis'],
 'ENFP': ['0 ½% due 30 Mat. Matched CDS', '0 ½% due 30 Basis'],
 'ENBCN': ['3 ⅛% due 29 Mat. Matched CDS', '3 ⅛% due 29 Basis'],
 'ENGIFP': ['1 ½% due 35 Mat. Matched CDS', '1 ½% due 35 Basis'],
 'ENIIM': ['1% due 34 Mat. Matched CDS', '1% due 34 Basis'],
 'EOANGR': ['5 ¾% due 33 Mat. Matched CDS', '5 ¾% due 33 Basis'],
 'EQNR': ['1 ⅝% due 35 Mat. Matched CDS', '1 ⅝% due 35 Basis'],
 'ET': ['4.9% due 35 Mat. Matched CDS', '4.9% due 35 Basis'],
 'EXC': ['5 ⅝% due 35 Mat. Matched CDS', '5 ⅝% due 35 Basis'],
 'EXPE': ['3 ¼% due 30 Mat. Matched CDS', '3 ¼% due 30 Basis'],
 'F': ['7.45% due 31 Mat. Matched CDS', '7.45% due 31 Basis'],
 'FCX': ['5.4% due 34 Mat. Matched CDS', '5.4% due 34 Basis'],
 'FDX': ['4 ¼% due 30 Mat. Matched CDS', '4 ¼% due 30 Basis'],
 'FE': ['4.55% due 30 Mat. Matched CDS', '4.55% due 30 Basis'],
 'FUMVFH': ['4 ½% due 33 Mat. Matched CDS', '4 ½% due 33 Basis'],
 'FRFP': ['4 ½% due 30 Mat. Matched CDS', '4 ½% due 30 Basis'],
 'EOFP': ['5 ½% due 31 Mat. Matched CDS', '5 ½% due 31 Basis'],
 'ASSGEN': ['2.124% due 30 Mat. Matched CDS', '2.124% due 30 Basis'],
 'GAP': ['3 ⅝% due 29 Mat. Matched CDS', '3 ⅝% due 29 Basis'],
 'GE': ['6 ¾% due 32 Mat. Matched CDS', '6 ¾% due 32 Basis'],
 'GFLCN': ['6 ⅝% due 32 Mat. Matched CDS', '6 ⅝% due 32 Basis'],
 'GIS': ['2 ⅞% due 30 Mat. Matched CDS', '2 ⅞% due 30 Basis'],
 'SOCGEN': ['1% due 30 Mat. Matched CDS', '1% due 30 Basis'],
 'GM': ['5% due 35 Mat. Matched CDS', '5% due 35 Basis'],
 'GS': ['6 ⅛% due 33 Mat. Matched CDS', '6 ⅛% due 33 Basis'],
 'GT': ['5 ¼% due 31 Mat. Matched CDS', '5 ¼% due 31 Basis'],
 'HAL': ['2.92% due 30 Mat. Matched CDS', '2.92% due 30 Basis'],
 'HBGCN': ['5 ⅝% due 29 Mat. Matched CDS', '5 ⅝% due 29 Basis'],
 'HD': ['2.7% due 30 Mat. Matched CDS', '2.7% due 30 Basis'],
 'HEIGR': ['3 ¾% due 32 Mat. Matched CDS', '3 ¾% due 32 Basis'],
 'HEIANA': ['2.02% due 32 Mat. Matched CDS', '2.02% due 32 Basis'],
 'HENKEL': ['0 ½% due 32 Mat. Matched CDS', '0 ½% due 32 Basis'],
 'HES': ['7 ⅞% due 29 Mat. Matched CDS', '7 ⅞% due 29 Basis'],
 'KHC': ['6 ¾% due 32 Mat. Matched CDS', '6 ¾% due 32 Basis'],
 'HON': ['1.95% due 30 Mat. Matched CDS', '1.95% due 30 Basis'],
 'HOTGR': ['4 ¼% due 30 Mat. Matched CDS', '4 ¼% due 30 Basis'],
 'HPQ': ['3.4% due 30 Mat. Matched CDS', '3.4% due 30 Basis'],
 'HSBC': ['0.77% due 31 Mat. Matched CDS', '0.77% due 31 Basis'],
 'IBM': ['5 ⅞% due 32 Mat. Matched CDS', '5 ⅞% due 32 Basis'],
 'INTNED': ['2 ½% due 30 Mat. Matched CDS', '2 ½% due 30 Basis'],
 'INTC': ['4% due 32 Mat. Matched CDS', '4% due 32 Basis'],
 'IRM': ['4 ⅞% due 29 Mat. Matched CDS', '4 ⅞% due 29 Basis'],
 'ISPIM': ['6.184% due 34 Mat. Matched CDS', '6.184% due 34 Basis'],
 'ITVLN': ['4 ¼% due 32 Mat. Matched CDS', '4 ¼% due 32 Basis'],
 'CCK': ['4 ½% due 30 Mat. Matched CDS', '4 ½% due 30 Basis'],
 'JNJ': ['6.95% due 29 Mat. Matched CDS', '6.95% due 29 Basis'],
 'JPM': ['4.452% due 29 Mat. Matched CDS', '4.452% due 29 Basis'],
 'KBH': ['4.8% due 29 Mat. Matched CDS', '4.8% due 29 Basis'],
 'KERFP': ['1 ⅞% due 30 Mat. Matched CDS', '1 ⅞% due 30 Basis'],
 'KMI': ['7.4% due 31 Mat. Matched CDS', '7.4% due 31 Basis'],
 'KPN': ['0 ⅞% due 32 Mat. Matched CDS', '0 ⅞% due 32 Basis'],
 'KR': ['7 ½% due 31 Mat. Matched CDS', '7 ½% due 31 Basis'],
 'KSS': ['4 ⅝% due 31 Mat. Matched CDS', '4 ⅝% due 31 Basis'],
 'L': ['6% due 35 Mat. Matched CDS', '6% due 35 Basis'],
 'LEN': ['5.2% due 30 Mat. Matched CDS', '5.2% due 30 Basis'],
 'LHAGR': ['4% due 30 Mat. Matched CDS', '4% due 30 Basis'],
 'LLOYDS': ['3 ⅛% due 30 Mat. Matched CDS', '3 ⅛% due 30 Basis'],
 'LMT': ['3.6% due 35 Mat. Matched CDS', '3.6% due 35 Basis'],
 'LNC': ['3.05% due 30 Mat. Matched CDS', '3.05% due 30 Basis'],
 'LOW': ['4 ½% due 30 Mat. Matched CDS', '4 ½% due 30 Basis'],
 'LUV': ['2 ⅝% due 30 Mat. Matched CDS', '2 ⅝% due 30 Basis'],
 'LW': ['4 ⅛% due 30 Mat. Matched CDS', '4 ⅛% due 30 Basis'],
 'LXSGR': ['0 ⅝% due 29 Mat. Matched CDS', '0 ⅝% due 29 Basis'],
 'BACRED': ['6 ½% due 33 Mat. Matched CDS', '6 ½% due 33 Basis'],
 'MBGGR': ['2% due 31 Mat. Matched CDS', '2% due 31 Basis'],
 'MBI': ['5.7% due 34 Mat. Matched CDS', '5.7% due 34 Basis'],
 'MCFP': ['0 ⅜% due 31 Mat. Matched CDS', '0 ⅜% due 31 Basis'],
 'MCD': ['2 ⅝% due 29 Mat. Matched CDS', '2 ⅝% due 29 Basis'],
 'MCK': ['5.1% due 33 Mat. Matched CDS', '5.1% due 33 Basis'],
 'MDC': ['3.85% due 30 Mat. Matched CDS', '3.85% due 30 Basis'],
 'MDLZ': ['2 ¾% due 30 Mat. Matched CDS', '2 ¾% due 30 Basis'],
 'MET': ['6 ½% due 32 Mat. Matched CDS', '6 ½% due 32 Basis'],
 'MWDP': ['1% due 31 Mat. Matched CDS', '1% due 31 Basis'],
 'MGM': ['6 ½% due 32 Mat. Matched CDS', '6 ½% due 32 Basis'],
 'MO': ['3.4% due 30 Mat. Matched CDS', '3.4% due 30 Basis'],
 'MPC': ['5.15% due 30 Mat. Matched CDS', '5.15% due 30 Basis'],
 'MSI': ['2.3% due 30 Mat. Matched CDS', '2.3% due 30 Basis'],
 'MTNA': ['3 ½% due 31 Mat. Matched CDS', '3 ½% due 31 Basis'],
 'MUR': ['6% due 32 Mat. Matched CDS', '6% due 32 Basis'],
 'NAVI': ['5 ⅝% due 33 Mat. Matched CDS', '5 ⅝% due 33 Basis'],
 'NEM': ['5 ⅞% due 35 Mat. Matched CDS', '5 ⅞% due 35 Basis'],
 'NEXIIM': ['3 ⅞% due 31 Mat. Matched CDS', '3 ⅞% due 31 Basis'],
 'NFLX': ['5 ⅜% due 29 Mat. Matched CDS', '5 ⅜% due 29 Basis'],
 'NGGLN': ['0.553% due 29 Mat. Matched CDS', '0.553% due 29 Basis'],
 'NOC': ['4.4% due 30 Mat. Matched CDS', '4.4% due 30 Basis'],
 'NOKIA': ['4 ⅜% due 31 Mat. Matched CDS', '4 ⅜% due 31 Basis'],
 'NRG': ['3 ⅝% due 31 Mat. Matched CDS', '3 ⅝% due 31 Basis'],
 'NSC': ['2.55% due 29 Mat. Matched CDS', '2.55% due 29 Basis'],
 'COOP': ['7 ⅛% due 32 Mat. Matched CDS', '7 ⅛% due 32 Basis'],
 'NWG': ['0.78% due 30 Mat. Matched CDS', '0.78% due 30 Basis'],
 'NWL': ['6 ⅝% due 29 Mat. Matched CDS', '6 ⅝% due 29 Basis'],
 'CPIPGR': ['1 ½% due 31 Mat. Matched CDS', '1 ½% due 31 Basis'],
 'OLN': ['5% due 30 Mat. Matched CDS', '5% due 30 Basis'],
 'OMC': ['2.45% due 30 Mat. Matched CDS', '2.45% due 30 Basis'],
 'ORAFP': ['8 ⅛% due 33 Mat. Matched CDS', '8 ⅛% due 33 Basis'],
 'ORCL': ['4.3% due 34 Mat. Matched CDS', '4.3% due 34 Basis'],
 'OVV': ['8 ⅛% due 30 Mat. Matched CDS', '8 ⅛% due 30 Basis'],
 'OXY': ['7 ½% due 31 Mat. Matched CDS', '7 ½% due 31 Basis'],
 'PARA': ['7 ⅞% due 30 Mat. Matched CDS', '7 ⅞% due 30 Basis'],
 'PFE': ['6 ½% due 34 Mat. Matched CDS', '6 ½% due 34 Basis'],
 'PG': ['5.8% due 34 Mat. Matched CDS', '5.8% due 34 Basis'],
 'PHIANA': ['2% due 30 Mat. Matched CDS', '2% due 30 Basis'],
 'PHM': ['7 ⅞% due 32 Mat. Matched CDS', '7 ⅞% due 32 Basis'],
 'PKG': ['3% due 29 Mat. Matched CDS', '3% due 29 Basis'],
 'PNLNA': ['4 ¾% due 31 Mat. Matched CDS', '4 ¾% due 31 Basis'],
 'POST': ['5 ½% due 29 Mat. Matched CDS', '5 ½% due 29 Basis'],
 'PPCGA': ['4 ⅝% due 31 Mat. Matched CDS', '4 ⅝% due 31 Basis'],
 'PRU': ['5 ¾% due 33 Mat. Matched CDS', '5 ¾% due 33 Basis'],
 'R': ['6.6% due 33 Mat. Matched CDS', '6.6% due 33 Basis'],
 'RABOBK': ['3 ⅞% due 32 Mat. Matched CDS', '3 ⅞% due 32 Basis'],
 'RCL': ['6 ¼% due 32 Mat. Matched CDS', '6 ¼% due 32 Basis'],
 'RIFP': ['0 ⅞% due 31 Mat. Matched CDS', '0 ⅞% due 31 Basis'],
 'RTX': ['7 ½% due 29 Mat. Matched CDS', '7 ½% due 29 Basis'],
 'RXLFP': ['5 ¼% due 30 Mat. Matched CDS', '5 ¼% due 30 Basis'],
 'SEE': ['6 ⅞% due 33 Mat. Matched CDS', '6 ⅞% due 33 Basis'],
 'SESGFP': ['4 ⅛% due 30 Mat. Matched CDS', '4 ⅛% due 30 Basis'],
 'SGOFP': ['1 ⅞% due 31 Mat. Matched CDS', '1 ⅞% due 31 Basis'],
 'SHAEFF': ['4 ½% due 30 Mat. Matched CDS', '4 ½% due 30 Basis'],
 'SHBASS': ['3 ¼% due 33 Mat. Matched CDS', '3 ¼% due 33 Basis'],
 'SHW': ['2.3% due 30 Mat. Matched CDS', '2.3% due 30 Basis'],
 'SO': ['2.65% due 29 Mat. Matched CDS', '2.65% due 29 Basis'],
 'SOLBBB': ['4 ¼% due 31 Mat. Matched CDS', '4 ¼% due 31 Basis'],
 'SRE': ['5 ½% due 33 Mat. Matched CDS', '5 ½% due 33 Basis'],
 'STANLN': ['1.2% due 31 Mat. Matched CDS', '1.2% due 31 Basis'],
 'STERV': ['0 ⅝% due 30 Mat. Matched CDS', '0 ⅝% due 30 Basis'],
 'STLA': ['1 ⅛% due 29 Mat. Matched CDS', '1 ⅛% due 29 Basis'],
 'SVC': ['4 ⅜% due 30 Mat. Matched CDS', '4 ⅜% due 30 Basis'],
 'SWEDA': ['3 ⅝% due 32 Mat. Matched CDS', '3 ⅝% due 32 Basis'],
 'T': ['4 ½% due 35 Mat. Matched CDS', '4 ½% due 35 Basis'],
 'TELNO': ['1 ¾% due 34 Mat. Matched CDS', '1 ¾% due 34 Basis'],
 'TELIAS': ['3 ½% due 33 Mat. Matched CDS', '3 ½% due 33 Basis'],
 'TGNA': ['5% due 29 Mat. Matched CDS', '5% due 29 Basis'],
 'TGT': ['6.35% due 32 Mat. Matched CDS', '6.35% due 32 Basis'],
 'THC': ['6 ⅞% due 31 Mat. Matched CDS', '6 ⅞% due 31 Basis'],
 'TITIM': ['7 ¾% due 33 Mat. Matched CDS', '7 ¾% due 33 Basis'],
 'TSN': ['4 ⅞% due 34 Mat. Matched CDS', '4 ⅞% due 34 Basis'],
 'TUIGR': ['1.95% due 31 Mat. Matched CDS', '1.95% due 31 Basis'],
 'UBS': ['0.65% due 29 Mat. Matched CDS', '0.65% due 29 Basis'],
 'UCGIM': ['2.731% due 32 Mat. Matched CDS', '2.731% due 32 Basis'],
 'URWFP': ['1 ⅜% due 30 Mat. Matched CDS', '1 ⅜% due 30 Basis'],
 'UNANA': ['1 ⅝% due 33 Mat. Matched CDS', '1 ⅝% due 33 Basis'],
 'UNH': ['4 ⅝% due 35 Mat. Matched CDS', '4 ⅝% due 35 Basis'],
 'UNP': ['3 ⅜% due 35 Mat. Matched CDS', '3 ⅜% due 35 Basis'],
 'VIEFP': ['6 ⅛% due 33 Mat. Matched CDS', '6 ⅛% due 33 Basis'],
 'VLO': ['7 ½% due 32 Mat. Matched CDS', '7 ½% due 32 Basis'],
 'VOD': ['2 ¾% due 34 Mat. Matched CDS', '2 ¾% due 34 Basis'],
 'VOVCAB': ['4 ¾% due 30 Mat. Matched CDS', '4 ¾% due 30 Basis'],
 'VZ': ['7 ¾% due 30 Mat. Matched CDS', '7 ¾% due 30 Basis'],
 'IPGIM': ['4 ⅞% due 30 Mat. Matched CDS', '4 ⅞% due 30 Basis'],
 'WHR': ['2.4% due 31 Mat. Matched CDS', '2.4% due 31 Basis'],
 'WMB': ['7 ½% due 31 Mat. Matched CDS', '7 ½% due 31 Basis'],
 'WMT': ['7.55% due 30 Mat. Matched CDS', '7.55% due 30 Basis'],
 'WY': ['7 ⅜% due 32 Mat. Matched CDS', '7 ⅜% due 32 Basis'],
 'YUM': ['4 ¾% due 30 Mat. Matched CDS', '4 ¾% due 30 Basis']}

In [7]:
bond_options   = sorted(col_meta, key=lambda c: col_meta[c][:3] + (c,))
market_options = list(df_bbg.columns)
options        = bond_options + market_options

issuers = ['All'] + sorted({col_meta[c][0] for c in bond_options}) + (['BBG'] if market_options else [])

def issuer_options(issuer):
    # Bond menus list '<bond> <metric>' without the issuer prefix, followed by the market series
    if issuer == 'All':
        return options
    if issuer == 'BBG':
        return market_options
    return [c.split(' ', 1)[1] for c in bond_options if col_meta[c][0] == issuer] + market_options

def resolve(issuer, value):
    # Turns a dropdown value back into a column of df
    if value in df.columns or issuer in ('All', 'BBG'):
        return value
    return f"{issuer} {value}"

def pick_defaults(issuer):
    # Returns (A, B) default menu values for an issuer
    if issuer in default_values_dict:
        a, b = default_values_dict[issuer]
        if f"{issuer} {a}" in df.columns and f"{issuer} {b}" in df.columns:
            return a, b
    target = pd.Timestamp(last_update) + pd.DateOffset(years=5)
    cands = bonds[(bonds["Issuer"] == issuer)].copy()
    cands = cands[cands["Label"].apply(lambda l: f"{l} Basis" in df.columns)]
    if cands.empty:
        return None
    live = cands[cands["ISIN"].map(latest("Basis")).notna()]
    cands = live if len(live) else cands
    label = cands.loc[(cands["Maturity"] - target).abs().idxmin(), "Label"]
    rest = label.split(" ", 1)[1]
    return f"{rest} Mat. Matched CDS", f"{rest} Basis"

print(f"{len(bond_options):,} bond series · {len(market_options)} market series · {len(issuers) - 1} issuers")

29,888 bond series · 13 market series · 344 issuers


In [8]:
LETTERS = list("abcdefghijkl")
SYMS = dict(zip(LETTERS, sp.symbols(" ".join(LETTERS))))
LETTER_RE = re.compile(r"(?<![A-Za-z_])([A-La-l])(?![A-Za-z_])")
TRANSFORMS = standard_transformations + (implicit_multiplication,)

def parse_expression(text):
    # 'b - a', '2a', 'log(b) - c', 'abs(a)/100' ... letters A-L map to the dropdowns
    text = (text or "").strip()
    if not text:
        return None, []
    expr = parse_expr(LETTER_RE.sub(lambda m: m.group(1).lower(), text),
                      local_dict=dict(SYMS), transformations=TRANSFORMS)
    used = sorted(str(s) for s in expr.free_symbols)
    unknown = [s for s in used if s not in SYMS]
    if unknown:
        raise ValueError(f"Unknown variable(s) {unknown}; use letters A-L")
    return expr, used

def evaluate(expr, used, data, col_map):
    fn = sp.lambdify([SYMS[s] for s in used], expr, modules="numpy")
    vals = np.asarray(fn(*[data[col_map[s]].to_numpy(dtype=float) for s in used]), dtype=float)
    vals = np.broadcast_to(vals, (len(data),)).copy()
    vals[~np.isfinite(vals)] = np.nan
    return pd.Series(vals, index=data.index)

def expression_label(text, col_map):
    def sub(m):
        prev = m.string[:m.start()].rstrip()[-1:]
        return ("*" if (prev.isdigit() or prev == ")") else "") + col_map[m.group(1).lower()]
    return LETTER_RE.sub(sub, text.strip())

def split_label(label, n=38):
    return '<br>'.join(label[i:i + n] for i in range(0, len(label), n))

def build_figure(issuer, letter_values, text1, text2, years):
    # letter_values: 12 dropdown values for A..L
    col_map = {l: resolve(issuer, v) for l, v in zip(LETTERS, letter_values)}
    expr1, used1 = parse_expression(text1)
    expr2, used2 = parse_expression(text2)
    if expr1 is None:
        raise ValueError("The horizontal-axis expression is empty")

    used_cols = list(dict.fromkeys(col_map[s] for s in used1 + used2))
    missing = [c for c in used_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Not in data: {missing}")
    data = df[used_cols]

    lab1 = expression_label(text1, col_map)
    lab2 = expression_label(text2, col_map) if expr2 is not None else ""

    full = pd.DataFrame({"Expression 1": evaluate(expr1, used1, data, col_map)})
    if expr2 is not None:
        full["Expression 2"] = evaluate(expr2, used2, data, col_map)

    # Forward-return columns: show today's value of the other expression as a reference line
    ref1 = ref2 = np.nan
    if expr2 is not None and 'FWD' in lab1 and 'FWD' not in lab2:
        ref2 = full["Expression 2"].dropna().iloc[-1]
    if expr2 is not None and 'FWD' in lab2 and 'FWD' not in lab1:
        ref1 = full["Expression 1"].dropna().iloc[-1]
    ref_date = df.index[-1].date()

    d = full.dropna()
    if years:
        d = d[d.index >= pd.Timestamp(datetime.now() - timedelta(days=int(years * 365))).normalize()]
    if d.empty:
        raise ValueError("No overlapping data for the selected series and period")

    x_all, last_dt = d["Expression 1"], d.index[-1].date()
    axis_style = dict(showline=True, linewidth=2, linecolor='black', mirror=True,
                      showgrid=True, gridcolor='LightGrey', gridwidth=1)

    if expr2 is None:
        fig = make_subplots(rows=1, cols=1, subplot_titles=(f'{lab1}: {x_all.iloc[-1]:.4f}',))
        fig.add_trace(go.Scatter(x=d.index, y=x_all, name=lab1, line=dict(color='blue')), row=1, col=1)
        fig.add_trace(go.Scatter(x=[d.index[0], d.index[-1]], y=[x_all.iloc[-1]] * 2, mode='lines',
                                 line=dict(dash='dash', color='blue'),
                                 name=f'Recent value of {lab1} is {x_all.iloc[-1]}'), row=1, col=1)
        fig.update_layout(yaxis=dict(zeroline=True, zerolinecolor='black'), hovermode='x unified',
                          plot_bgcolor='white', paper_bgcolor='white',
                          legend=dict(orientation="h", yanchor="bottom", xanchor="center", x=0.5, y=1.05),
                          height=800, width=1100)
        fig.update_xaxes(title='Date', row=1, col=1, **axis_style)
        fig.update_yaxes(title=split_label(lab1), side='left', row=1, col=1, **axis_style)
        return fig

    y_all = d["Expression 2"]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.1,
                        specs=[[{"secondary_y": False}], [{"secondary_y": True}]],
                        subplot_titles=(f'Scatter Plot from {d.index[0].date()} to {last_dt}',
                                        f'{lab1}: {x_all.iloc[-1]:.4f}   &   {lab2}: {y_all.iloc[-1]:.4f}'))

    def pts(sl, name, color):
        fig.add_trace(go.Scatter(x=x_all.iloc[sl], y=y_all.iloc[sl], name=name, mode='markers',
                                 marker=dict(color=color), text=[str(i.date()) for i in d.index[sl]],
                                 hovertemplate='%{text}<br>x=%{x:.4f}<br>y=%{y:.4f}'), row=1, col=1)

    pts(slice(-63, -1),   '< 3 months',            'green')
    pts(slice(-252, -63), '> 3 months & < 1 year', 'blue')
    pts(slice(None, -252),'> 1 year',              'gray')
    fig.add_trace(go.Scatter(x=[x_all.iloc[-1]], y=[y_all.iloc[-1]], name=f'Value as of {last_dt}',
                             mode='markers', marker=dict(color='red', size=10)), row=1, col=1)

    if len(d) > 3 and x_all.nunique() > 2:
        coeffs = np.polyfit(x_all, y_all, 2)
        poly = np.poly1d(coeffs)
        x_poly = np.linspace(x_all.min(), x_all.max(), 500)
        y_poly = poly(x_poly)
        y_pred = poly(x_all)
        r2 = r2_score(y_all, y_pred)
        se = np.sqrt(np.sum((y_all - y_pred) ** 2) / (len(x_all) - len(coeffs)))
        eq = f"y = {coeffs[0]:.4f}*x² + {coeffs[1]:.4f}*x + {coeffs[2]:.4f}"
        fig.add_trace(go.Scatter(x=x_poly, y=y_poly, name=f'Polynomial Fit<br>R² = {r2:.3f}<br>{eq}',
                                 mode='lines', line=dict(color='orange')), row=1, col=1)
        fig.add_trace(go.Scatter(x=x_poly, y=y_poly + 2 * se, name='Upper Confidence Band', mode='lines',
                                 line=dict(color='orange', dash='dash')), row=1, col=1)
        fig.add_trace(go.Scatter(x=x_poly, y=y_poly - 2 * se, name='Lower Confidence Band', mode='lines',
                                 fill='tonexty', fillcolor='rgba(255,165,0,0.15)',
                                 line=dict(color='orange', dash='dash')), row=1, col=1)

    if not np.isnan(ref2):
        fig.add_trace(go.Scatter(x=x_all, y=[ref2] * len(x_all), mode='lines', line=dict(color='red', width=1),
                                 name=f'{lab2} on {ref_date} was {ref2}'), row=1, col=1)
    if not np.isnan(ref1):
        fig.add_trace(go.Scatter(x=[ref1] * len(y_all), y=y_all, mode='lines', line=dict(color='red', width=1),
                                 name=f'{lab1} on {ref_date} was {ref1}'), row=1, col=1)

    fig.add_trace(go.Scatter(x=d.index, y=x_all, name=lab1, line=dict(color='blue')),
                  row=2, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=[d.index[0], d.index[-1]], y=[x_all.iloc[-1]] * 2, mode='lines',
                             line=dict(dash='dash', color='blue'), name=f'Recent value of {lab1} is {x_all.iloc[-1]}'),
                  row=2, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=d.index, y=y_all, name=lab2, line=dict(color='green')),
                  row=2, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=[d.index[0], d.index[-1]], y=[y_all.iloc[-1]] * 2, mode='lines',
                             line=dict(dash='dash', color='green'), name=f'Recent value of {lab2} is {y_all.iloc[-1]}'),
                  row=2, col=1, secondary_y=True)

    fig.update_layout(hovermode='x unified', plot_bgcolor='white', paper_bgcolor='white',
                      legend=dict(orientation="h", yanchor="bottom", xanchor="center", x=0.5, y=1.05),
                      height=1900, width=1100)
    fig.update_xaxes(title=split_label(lab1), row=1, col=1, **axis_style)
    fig.update_yaxes(title=split_label(lab2), row=1, col=1, secondary_y=False, **axis_style)
    fig.update_xaxes(title='Date', row=2, col=1, **axis_style)
    fig.update_yaxes(title=split_label(lab1), side='left', row=2, col=1, secondary_y=False,
                     title_font=dict(color='blue'), **axis_style)
    fig.update_yaxes(title=split_label(lab2), side='right', row=2, col=1, secondary_y=True,
                     showline=True, linewidth=2, linecolor='black', mirror=True, title_font=dict(color='green'))
    return fig

In [9]:
# Starts on the first issuer rather than 'All' ('All' loads every series into 12 menus and is slow)
dropdown1   = Dropdown(options=issuers, value=issuers[1], description='Issuer')
letter_dds  = [Dropdown(options=[], description=f'{L.upper()}:') for L in LETTERS]
years_box   = widgets.FloatText(value=5, description='Years:', step=0.01)

display(VBox([HBox([dropdown1])] +
             [HBox(letter_dds[i:i + 3]) for i in range(0, 12, 3)] +
             [HBox([years_box])]))

expression_input1 = Text(description='Horizontal Axis:', value='a')
expression_input2 = Text(description='Vertical Axis:',   value='b')
submit_button = Button(description="Submit")
output = Output()
display(expression_input1, expression_input2, submit_button, output)

def on_button_clicked(_):
    with output:
        clear_output()
        try:
            fig = build_figure(dropdown1.value, [dd.value for dd in letter_dds],
                               expression_input1.value, expression_input2.value, years_box.value)
            fig.show()
        except Exception as err:
            print(f"Could not plot: {err}")

def update_dropdowns(*_):
    issuer = dropdown1.value
    opts = issuer_options(issuer)
    for dd in letter_dds:
        dd.options = opts
    defaults = pick_defaults(issuer) if issuer not in ('All', 'BBG') else None
    if defaults:
        letter_dds[0].value, letter_dds[1].value = defaults

submit_button.on_click(on_button_clicked)
dropdown1.observe(update_dropdowns, names='value')
update_dropdowns()

Text(value='a', description='Horizontal Axis:')

Text(value='b', description='Vertical Axis:')

Button(description='Submit', style=ButtonStyle())

Output()